### ENSEMBLE METHODS

Combining models from different families through stacking and voting
1. Voting - combine predictions by averaging (soft) or majority vote (hard) - using trained models saved in 
2. Stacking - train a meta-model that learns how to combine the base models’ predictions.

Compute out-of-fold predictions for all candidate models.
Check the correlation matrix of those predictions. Drop models that are highly correlated with a better one.
Keep the top few per family, or use greedy forward selection (add the model that most improves ensemble CV score; repeat until no gain).
Compare: best single model vs. voting vs. stacking, using the same CV. Use a held-out test set for the final comparison.

||Voting|Stacking|
| :--- | :---: | ---: |
|Complexity |Low|Higher|
|Overfitting risk|Low|Moderate (more models → more risk)|
|Gain|Reliable, modest|	Often larger if models are diverse|
|Best when|	Models are similarly strong|	Models have different strengths

Importing all predictions on validation

everyone validate on sarcasm random + sarcasm subreddit, sarcasm temporal and news headlines 
voting is on validation 

shouldnt be on test data - unseen
use validation data 

need probability score and actual predictions for each row 

4 models results r quite similar -> can use voting
subreddit - make it worse -> explanation: affect it's ability to ?? 

lgbm - for prediction
ebm for explanation
shap for feature explanation

0.001 - subreddit very low contribution to prediction
0.001 - parent thread contribution 

cannot test data 
author say overlap between train and test data, same context have diff meaning

lgbm xgboost log_reg , linear svm - voting

all voting and stacking done with the manual way - not using votingclassifier model becuase its from prediction results not from models






In [19]:
#import libraries
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as  np
from scipy.stats import mode
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import cross_val_predict, StratifiedKFold

In [ ]:
#import validation dataset 
#sarc_random_validation
#define routes


#validation and test data
val  = pd.read_parquet("../../data/splits/primary/sarc_random/val.parquet")
test  = pd.read_parquet("../../data/splits/primary/sarc_random/test.parquet")
y_val = val['label']
y_test = test['label']

#sarcasm_random validation predictions (probabilties)
#lite gbm /xgboost
val_lgbm = "../../results/gbdt/ensemble_predictions/sarc_random__val.parquet"
#margin-based (linear svm)
val_margin = "../../results/margin-based/ensemble_predictions/sarc_random__val.parquet"
#log_reg
val_logreg= "../../results/nb_logreg/ensemble_predictions/sarc_random__val.parquet"

#sarcasm_random test predictions (probabilties)
#lite gbm /xgboost
test_lgbm = "../../results/gbdt/ensemble_predictions/sarc_random__test.parquet"
#margin-based (linear svm)
test_margin = "../../results/margin-based/ensemble_predictions/sarc_random__test.parquet"
#log_reg
test_logreg= "../../results/nb_logreg/ensemble_predictions/sarc_random__test.parquet"


#sarc_subreddit probability predictions
subreddit_val_lgbm = "../../results/gbdt/ensemble_predictions/sarc_subreddit__val.parquet" #lgbm
subreddit_test_lgbm = "../../results/gbdt/ensemble_predictions/sarc_subreddit__test.parquet"

subreddit_val_logreg= "../../results/nb_logreg/ensemble_predictions/sarc_subreddit__val.parquet" 
subreddit_test_logreg= "../../results/nb_logreg/ensemble_predictions/sarc_subreddit__test.parquet" #logreg

subreddit_val_margin = "../../results/margin-based/ensemble_predictions/sarc_random__val.parquet" #margin based
subreddit_test_margin = "../../results/margin-based/ensemble_predictions/sarc_random__test.parquet" 

#sarc_temporal
temporal_val_lgbm = "../../results/gbdt/ensemble_predictions/sarc_temporal__val.parquet" #lgbm
temporal_test_lgbm = "../../results/gbdt/ensemble_predictions/sarc_temporal__test.parquet"

temporal_val_logreg= "../../results/nb_logreg/ensemble_predictions/sarc_temporal__val.parquet" 
temporal_test_logreg= "../../results/nb_logreg/ensemble_predictions/sarc_temporal__test.parquet" #logreg

temporal_val_margin = "../../results/margin-based/ensemble_predictions/sarc_temporal__val.parquet" #margin based
temporal_test_margin = "../../results/margin-based/ensemble_predictions/sarc_temporal__test.parquet" 

#figlang_twitter
figlang_twitter_lgbm = "../../results/gbdt/ensemble_predictions/figlang_twitter__test.parquet"
figlang_twitter_logreg= "../../results/nb_logreg/ensemble_predictions/figlang_twitter__test.parquet" #logreg
figlang_twitter_margin = "../../results/margin-based/ensemble_predictions/figlang_twitter___test.parquet"   #margin based

#figlang_reddit
figlang_reddit_lgbm = "../../results/gbdt/ensemble_predictions/figlang_reddit__test.parquet"
figlang_reddit_logreg= "../../results/nb_logreg/ensemble_predictions/figlang_reddit__test.parquet" #logreg
figlang_reddit_margin = "../../results/margin-based/ensemble_predictions/figlang_reddit___test.parquet"   #margin based

#news_headlines
newsheadlines_lgbm = "../../results/gbdt/ensemble_predictions/news_headlines__test.parquet"
newsheadlines_logreg= "../../results/nb_logreg/ensemble_predictions/news_headlines__test.parquet" #logreg
newsheadlines_margin = "../../results/margin-based/ensemble_predictions/news_headlines___test.parquet"   #margin based

#tweeteval_irony
#news_headlines
tweeteval_lgbm = "../../results/gbdt/ensemble_predictions/tweeteval_irony__test.parquet"
tweeteval_logreg= "../../results/nb_logreg/ensemble_predictions/tweeteval_irony__test.parquet" #logreg
tweeteval_margin = "../../results/margin-based/ensemble_predictions/tweeteval_irony___test.parquet"   #margin based

df = pd.read_parquet(subreddit_val_lgbm)
display(df)

#model names endings in columns
lgbm_model = 'reply_tfidf_surface'
xgb_model = 'reply_tfidf_surface'
lr_model = 'reply_tfidf_surface'
svm_model = 'svm_tfidf_lex_surf'

display(pd.read_parquet(val_lgbm))





,row_id,row_position,label,sarc_subreddit__lgbm_reply_tfidf,sarc_subreddit__lgbm_reply_tfidf_surface,sarc_subreddit__xgb_reply_tfidf,sarc_subreddit__xgb_reply_tfidf_surface
0,8684f126dd72506028ac9a8561208d91,0,0,0.098881,0.102032,0.128582,0.080833
1,ebb64764d1fff111927e9f6836c49aad,1,0,0.248388,0.269309,0.313396,0.328555
2,1ad3b1ed384e722f477a12894cdda1c5,2,0,0.440454,0.622033,0.435739,0.424888
3,1049a818926882961689a8867de89602,3,0,0.250224,0.194810,0.285878,0.252661
4,9b931bfdd0c0aa07697733c968def0bf,4,0,0.386554,0.369191,0.394082,0.396527
...,...,...,...,...,...,...,...
135118,546b0bfe6cc64a0b4eadc7b73ba1c88f,135118,1,0.944702,0.943444,0.916125,0.938240
135119,7bccef68177af56d32c8b0c2aaaafb66,135119,1,0.911136,0.908983,0.903309,0.934739
135120,e7d8450902798e0a560c50650ca1ff8e,135120,1,0.618087,0.639425,0.681469,0.718837
135121,b7c26f7d57cbcc16dcc93ac743ff207f,135121,0,0.205533,0.221221,0.201609,0.208460


,row_id,row_position,label,sarc_random__lgbm_reply_tfidf,sarc_random__lgbm_reply_tfidf_surface,sarc_random__xgb_reply_tfidf,sarc_random__xgb_reply_tfidf_surface
0,8c5622f4b8edbd4fc1bb29d1a2739d90,0,0,0.349346,0.379181,0.549932,0.441270
1,ef194a1d7209570a5779540806d0be7f,1,0,0.520090,0.425580,0.462667,0.385260
2,0b6b7d6b88a0165663241c9eb01dfc9e,2,0,0.099563,0.101146,0.120174,0.116042
3,f454a51a9c08b6bb9021364fe055076d,3,0,0.095299,0.079013,0.185215,0.104300
4,39d85935396f36f884644344b6ec9121,4,0,0.191140,0.302087,0.179507,0.158919
...,...,...,...,...,...,...,...
137061,1a110007fb9569efbcbcdbc48ef7c54f,137061,1,0.729605,0.711259,0.718466,0.770864
137062,cd6a3e9182e07a0b4efc4e210edfd344,137062,1,0.966070,0.937673,0.938973,0.884863
137063,1e71c35e434299170bf6014f168b8a8d,137063,1,0.305158,0.588000,0.333046,0.579210
137064,c64bcf3dda6243a1c9c725300b1ded4f,137064,1,0.315640,0.268119,0.340868,0.314942


## Hard Voting
- using the majority vote of all 4 models - lgbm, xgboost, lr, svm


In [40]:

# def hard_vote(sources, id_path):
#     """
#     sources: dict {model_name: (path, col_name)}
#     id_path: parquet file to take 'row_id' from
#     """
#     def get_prediction(path, col_name, threshold):
#         df = pd.read_parquet(path)
#         probs = df.loc[:, df.columns.str.endswith(col_name)]
#         preds = np.array([1 if p >= threshold else 0 for p in probs])
#         return preds

#     names = list(sources)

#     final_pred = pd.DataFrame({'row_id': pd.read_parquet(id_path)['row_id']})

#     for name, (path, col, threshold) in sources.items():
#         final_pred[name] = get_prediction(path, col, threshold)

#     # majority voting
#     final_pred['hardvoting_pred'] = final_pred[names].mode(axis=1)[0] #default to 0 if number of 0s and 1s is the same
#     return final_pred

def hard_vote(sources, id_path):
    """
    sources: dict {model_name: (path, col_name, threshold)}
    id_path: parquet file to take 'row_id' from
    """
    def get_prediction(path, col_name, threshold):
        df = pd.read_parquet(path)
        
        # 1. Dynamically locate the exact column matching the suffix
        matched_cols = df.columns[df.columns.str.endswith(col_name)]
        if len(matched_cols) == 0:
            raise KeyError(f"No column ending with '{col_name}' found in {path}")
            
        # 2. Extract the first matching column as a Series / 1D array
        probs = df[matched_cols[0]]
        
        # 3. Vectorized comparison (Blazing fast compared to list comprehension)
        preds = (probs >= threshold).astype(int).values
        return preds

    names = list(sources.keys())

    # Load initial row IDs
    final_pred = pd.DataFrame({'row_id': pd.read_parquet(id_path)['row_id']})

    # Note the unpacking matching the 3-element tuple in your loop: (path, col, threshold)
    for name, (path, col, threshold) in sources.items():
        final_pred[name] = get_prediction(path, col, threshold)

    # 4. Safe majority voting with a guaranteed fallback to 0 for ties
    # mode(axis=1) can return multiple columns if there's a tie, so we take the first column [0]
    final_pred['hardvoting_pred'] = final_pred[names].mode(axis=1)[0].astype(int)
    
    return final_pred



In [51]:
#hard voting on validation data 
final_pred_val = hard_vote({
    'lgbm':       (val_lgbm,   lgbm_model, 0.5),
    'xgboost':    (val_lgbm,   xgb_model, 0.5),
    'lr':         (val_logreg, lr_model, 0.510238),
    'linear_svm': (val_margin, svm_model, 0.5)},  
    id_path=val_logreg
)

#hard voting on test data - using lgbm, xgboost, lr, linear_svm
final_pred_test = hard_vote({
    'lgbm':       (test_lgbm,   lgbm_model, 0.5),
    'xgboost':    (test_lgbm,   xgb_model, 0.5),
    'lr':         (test_logreg, lr_model, 0.510238),
    'linear_svm': (test_margin, svm_model, 0.5)},  
    id_path=test_logreg
)


## Soft Voting
- using probability of prediction for each model to determine weights of each model in prediction
- weights calculated from validation predictions and then applied on test data 


In [ ]:

#plain soft voting - equal weight
def equal_weight_soft_vote(test_sources, id_path, threshold=0.5):
    """
    sources: dict {model_name: (path, col_name)}
    id_path: parquet file to take 'row_id' from
    threshold: cutoff applied to the averaged probability
    """
    ids = pd.read_parquet(id_path)['row_id']
    final_pred = pd.DataFrame({'row_id': ids})
    for name, (path, col) in test_sources.items():
        df = pd.read_parquet(path)
        matched_cols = df.columns[df.columns.str.endswith(col)]
        if len(matched_cols) == 0:
            raise KeyError(f"No column ending with '{col}' found in {path}")
            
        # 2. Extract the first matching column as a Series / 1D array
        probs = df[matched_cols[0]]
        final_pred[name] = probs.values

    names = list(test_sources)
    final_pred['mean_prob'] = final_pred[names].mean(axis=1)
    final_pred['softvoting_pred'] = (final_pred['mean_prob'] >= threshold).astype(int)
    return final_pred



#adjust weights based on probabilities
# def soft_vote(val_sources, test_sources, y_val, id_path=None, threshold=0.5):
#     """
#     val_sources / test_sources: dict {model_name: (path, col_name)}
#         (same model names, in the same order, for both)
#     y_val: true validation labels
#     id_path: optional test parquet file to take 'row_id' from
#     Returns (result_df, scores, weights)
#     """
#     def get_prob(path, col):
#         matched_cols = df.columns[df.columns.str.endswith(col)]
#         if len(matched_cols) == 0:
#                     raise KeyError(f"No column ending with '{col}' found in {path}")
                    
#         # 2. Extract the first matching column as a Series / 1D array
#         probs = df[matched_cols[0]]
#         return pd.read_parquet(path)[col].values

#     names = list(val_sources)
#     if list(test_sources) != names:
#         raise ValueError("val_sources and test_sources must have the same model names in the same order")

#     #calculate weights from validation ROC-AUC
#     scores = np.array([
#         roc_auc_score(y_val, get_prob(*val_sources[n])) for n in names
#     ])
#     weights = (scores - 0.5) / (scores - 0.5).sum()

#     #apply weights to test probabilities
#     P_test = np.column_stack([get_prob(*test_sources[n]) for n in names])
#     final_prob = np.average(P_test, axis=1, weights=weights)
#     softvoting_preds = (final_prob >= threshold).astype(int)

#     #predict for test
#     result = pd.DataFrame(P_test, columns=names)
#     if id_path is not None:
#         result.insert(0, 'row_id', pd.read_parquet(id_path)['row_id'].values)
#     result['softvoting_prob'] = final_prob
#     result['softvoting_pred'] = softvoting_preds

#     return result, pd.Series(scores, index=names), pd.Series(weights, index=names)


def soft_vote(val_sources, test_sources, y_val, id_path=None, threshold=0.5):
    """
    val_sources / test_sources: dict {model_name: (path, col_name)}
        (same model names, in the same order, for both)
    y_val: true validation labels
    id_path: optional test parquet file to take 'row_id' from
    Returns (result_df, scores, weights)
    """
    def get_prob(path, col):
        df = pd.read_parquet(path)
        matched_cols = df.columns[df.columns.str.endswith(col)]
        if len(matched_cols) == 0:
            raise KeyError(f"No column ending with '{col}' found in {path}")
                    
        return df[matched_cols[0]].values

    names = list(val_sources.keys())
    if list(test_sources.keys()) != names:
        raise ValueError("val_sources and test_sources must have the same model names in the same order")

    # Calculate weights from validation ROC-AUC
    scores = np.array([
        roc_auc_score(y_val, get_prob(*val_sources[n])) for n in names
    ])
    
    # Handle edge case where all scores might be <= 0.5 to prevent division by zero
    score_diffs = scores - 0.5
    if score_diffs.sum() == 0:
        weights = np.ones(len(names)) / len(names)  # Fall back to equal weights
    else:
        weights = score_diffs / score_diffs.sum()

    # Apply weights to test probabilities
    P_test = np.column_stack([get_prob(*test_sources[n]) for n in names])
    
    # Use np.average with your calculated weights array
    final_prob = np.average(P_test, axis=1, weights=weights)
    softvoting_preds = (final_prob >= threshold).astype(int)

    # Predict for test data
    result = pd.DataFrame(P_test, columns=names)
    if id_path is not None:
        result.insert(0, 'row_id', pd.read_parquet(id_path)['row_id'].values)
        
    result['softvoting_prob'] = final_prob
    result['softvoting_pred'] = softvoting_preds

    return result, pd.Series(scores, index=names), pd.Series(weights, index=names)

In [55]:
#plain soft voting - equal weight for all models
softvote_results, scores, weights = soft_vote(
    val_sources={
        'lgbm':       (val_lgbm,   lgbm_model),
        'xgboost':    (val_lgbm,   xgb_model),
        'lr':         (val_logreg, lr_model),
        'linear_svm': (val_margin, svm_model),
    },
    test_sources={
        'lgbm':       (test_lgbm,   lgbm_model),
        'xgboost':    (test_lgbm,   xgb_model),   
        'lr':         (test_logreg, lr_model),
        'linear_svm': (test_margin, svm_model),
    },
    y_val=y_val,
    id_path=test_logreg,
)


equalweight_results = equal_weight_soft_vote(
    test_sources={
        'lgbm':       (test_lgbm,   lgbm_model),
        'xgboost':    (test_lgbm,   xgb_model),   
        'lr':         (test_logreg, lr_model),
        'linear_svm': (test_margin, svm_model),
    },
    id_path=test_logreg,
)

#try dropping lgbm model since it is highly correlated to xgboost
softvote_results_withoutxgb, scores, weights = soft_vote(
    val_sources={
        'xgboost':    (val_lgbm,   xgb_model),
        'lr':         (val_logreg, lr_model),
        'linear_svm': (val_margin, svm_model),
    },
    test_sources={ 
        'xgboost':    (test_lgbm,   xgb_model),
        'lr':         (test_logreg, lr_model),
        'linear_svm': (test_margin, svm_model),
    },
    y_val=y_val,
    id_path=test_logreg,
)



### Stacking

Uses logistic regression as the meta model to fit 

```
logit(final) = b + w1·z1 + w2·z2 + w3·z3 + w4·z4
```
z is the base model prediction   
w are weights learned from predictions  
if a model is similar to another model, it gets a weight near zero or even negative (which voting cannot do)



In [69]:
#Stacking with log regression

# def load_probs(sources):
#     cols = {}
#     for name, (path, col) in sources.items():
#         # 1. Read only the schema header first to find the exact column name
#         all_cols = pd.read_parquet(path, iterator=True).schema.names
#         matched_cols = [c for c in all_cols if c.endswith(col)]
        
#         if len(matched_cols) == 0:
#             raise KeyError(f"No column ending with '{col}' found in {path}")
            
#         # 2. Extract using the full column name and pull its 1D numpy array values
#         target_col = matched_cols
#         cols[name] = pd.read_parquet(path, columns=[target_col])[target_col].values
        
#     return pd.DataFrame(cols)

def get_prob(sources):
    cols={}
    for name, (path, col) in sources.items():
        df = pd.read_parquet(path)
        matched_cols = df.columns[df.columns.str.endswith(col)]
        if len(matched_cols) == 0:
            raise KeyError(f"No column ending with '{col}' found in {path}")

        cols[name] = df[matched_cols[0]].values
        return pd.DataFrame(cols)

def to_logit(P, eps=1e-6):
    P = P.clip(eps, 1 - eps)
    return np.log(P / (1 - P))

def stacking(val_sources, test_sources, y_val):

    P_val = get_prob(val_sources)
    P_test = get_prob(test_sources)

    Z_val, Z_test = to_logit(P_val), to_logit(P_test)

    #tune different values of regularization
    meta = LogisticRegressionCV(Cs=[0.01, 0.1, 1, 10], cv=5, scoring='neg_log_loss', max_iter=1000)

    # out-of-fold estimate within validation
    cv = StratifiedKFold(5, shuffle=True, random_state=42)
    p_val_oof = cross_val_predict(meta, Z_val, y_val, cv=cv, method='predict_proba')[:, 1]

    # choose threshold on the out-of-fold predictions using f1 scores
    thresholds = np.linspace(0.05, 0.95, 91)
    f1s = [f1_score(y_val, p_val_oof >= t) for t in thresholds]
    best_t = thresholds[int(np.argmax(f1s))]
    print('best threshold', best_t, 'val F1 (OOF)', max(f1s))

    # fit on all of validation, apply once to test
    meta.fit(Z_val, y_val)
    print(dict(zip(Z_val.columns, meta.coef_[0].round(3))))   # model weights
    print('best c :', meta.C_[0])
    print("log correlation :", Z_val.corr().round(3))
    p_test = meta.predict_proba(Z_test)[:, 1]
    test_pred = (p_test >= best_t).astype(int)

    return meta, best_t, p_test, test_pred



In [71]:
meta, best_t, p_test, stacking_test_pred= stacking(
    val_sources={
        'lgbm':       (val_lgbm,   lgbm_model),
        'xgboost':    (val_lgbm,   xgb_model),
        'logreg':         (val_logreg, lr_model),
        'linear_svm': (val_margin, svm_model),
    },
    test_sources={
        'lgbm':       (test_lgbm,   lgbm_model),
        'xgboost':    (test_lgbm,   xgb_model),   
        'logreg':         (test_logreg, lr_model),
        'linear_svm': (test_margin, svm_model),
    },
    y_val=y_val,
)

#stacking without lgbm 
meta, best_t, p_test, stacking_test_pred_withoutxgb= stacking(
    val_sources={
        'lgbm':       (val_lgbm,   lgbm_model),
        'logreg':     (val_logreg, lr_model),
        'linear_svm': (val_margin, svm_model),
    },
    test_sources={
        'lgbm':       (test_lgbm,   lgbm_model),
        'logreg':     (test_logreg, lr_model), 
        'linear_svm': (test_margin, svm_model), 
    },
    y_val=y_val,
)

print(stacking_test_pred)
print(stacking_test_pred_withoutxgb)

#strong log correlation between 
#xgboost and lgbm
#logreg and linear_svm 

#weight for linear_svm model is very high - around 0.424 
#weight for logreg is negative
#almost similar weight for xgboost and lgbm


best threshold 0.37999999999999995 val F1 (OOF) 0.7413015881240447
{'lgbm': np.float64(0.973)}
best c : 0.1
log correlation :       lgbm
lgbm   1.0
best threshold 0.37999999999999995 val F1 (OOF) 0.7413015881240447
{'lgbm': np.float64(0.973)}
best c : 0.1
log correlation :       lgbm
lgbm   1.0
[0 1 0 ... 1 1 1]
[0 1 0 ... 1 1 1]


## Evaluation


In [72]:
def evaluate(y, p):
    pred = p 
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"roc_auc": float(roc_auc_score(y, p)),
            "pr_auc": float(average_precision_score(y, p)),
            "pr_auc_definition": "average_precision",
            "accuracy": float(accuracy_score(y, pred)),
            "precision": float(precision_score(y, pred, zero_division=0)),
            "recall": float(recall_score(y, pred, zero_division=0)),
            "f1": float(f1_score(y, pred, zero_division=0)),
            "brier_score": float(brier_score_loss(y, p)),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

In [76]:

#hard voting evaluation on test 
print("Hard Voting with LightGBM, XGBoost, Linear Regression and Linear SVM")
display(evaluate(y_test, final_pred_test['hardvoting_pred']))

#soft voting evaluation 
print("Soft Voting with LightGBM, XGBoost, Linear Regression and Linear SVM")
display(evaluate(y_test, softvote_results['softvoting_pred']))

#soft voting without lgbm 
print("Soft Voting with LightGBM,Linear Regression and Linear SVM")
display(evaluate(y_test, softvote_results_withoutxgb['softvoting_pred']))
#dropping lgbm makes the results worse

#plain soft voting - equal weight evaluation
print("Equal Weight Voting with LightGBM, XGBoost, Linear Regression and Linear SVM")
display(evaluate(y_test, equalweight_results['softvoting_pred']))

#stacking
print("Stacking with LightGBM, XGBoost, Linear Regression and Linear SVM") 
display(evaluate(y_test, stacking_test_pred))

#stacking without xgb
print("Stacking with LightGBM  Linear Regression and Linear SVM") 
display(evaluate(y_test, stacking_test_pred_withoutxgb))

#removing models with high correlation to other models does not necessarily improve the model
#in this case, removing the models made results worse


Hard Voting with LightGBM, XGBoost, Linear Regression and Linear SVM


{'roc_auc': 0.7339565325750073,
 'pr_auc': 0.6899154547443531,
 'pr_auc_definition': 'average_precision',
 'accuracy': 0.7323459672418196,
 'precision': 0.7827533525717195,
 'recall': 0.6587428571428572,
 'f1': 0.7154138546272594,
 'brier_score': 0.26765403275818045,
 'tn': 54267,
 'fp': 12798,
 'fn': 23888,
 'tp': 46112}

Soft Voting with LightGBM, XGBoost, Linear Regression and Linear SVM


{'roc_auc': 0.7335766196973087,
 'pr_auc': 0.6849191918595829,
 'pr_auc_definition': 'average_precision',
 'accuracy': 0.7327034618611608,
 'precision': 0.7621681937481337,
 'recall': 0.6928,
 'f1': 0.7258304709307112,
 'brier_score': 0.26729653813883925,
 'tn': 51932,
 'fp': 15133,
 'fn': 21504,
 'tp': 48496}

Soft Voting with LightGBM,Linear Regression and Linear SVM


...

{'roc_auc': 0.7320490797840049,
 'pr_auc': 0.6836877808971575,
 'pr_auc_definition': 'average_precision',
 'accuracy': 0.7311275672126364,
 'precision': 0.7617626154939587,
 'recall': 0.6890142857142857,
 'f1': 0.7235644901173911,
 'brier_score': 0.26887243278736367,
 'tn': 51981,
 'fp': 15084,
 'fn': 21769,
 'tp': 48231}

Stacking with LightGBM, XGBoost, Linear Regression and Linear SVM


{'roc_auc': 0.7101228041026296,
 'pr_auc': 0.6507782714177306,
 'pr_auc_definition': 'average_precision',
 'accuracy': 0.7125086637726626,
 'precision': 0.6812049134693974,
 'recall': 0.8215428571428571,
 'f1': 0.7448209764216007,
 'brier_score': 0.2874913362273374,
 'tn': 40152,
 'fp': 26913,
 'fn': 12492,
 'tp': 57508}

Stacking with LightGBM  Linear Regression and Linear SVM


{'roc_auc': 0.7101228041026296,
 'pr_auc': 0.6507782714177306,
 'pr_auc_definition': 'average_precision',
 'accuracy': 0.7125086637726626,
 'precision': 0.6812049134693974,
 'recall': 0.8215428571428571,
 'f1': 0.7448209764216007,
 'brier_score': 0.2874913362273374,
 'tn': 40152,
 'fp': 26913,
 'fn': 12492,
 'tp': 57508}

In [29]:
#check correlation of lgbm and xgboost model 
def get_prob(path, col_name):
        return pd.read_parquet(path)[col_name].values

P_val = np.column_stack([get_prob(val_lgbm, lgbm_model), get_prob(val_lgbm , xgb_model)
                         , get_prob(val_logreg, lr_model), get_prob(val_margin, svm_model)])

print("Correlation between models")
print(pd.DataFrame(P_val, columns = ['lgbm', 'xgb', 'lr', 'svm']).corr().round(3))

#high corration between lgbm and xgboost models -> errors might be highly correlated
#maybe can use another model


Correlation between models
       lgbm    xgb     lr    svm
lgbm  1.000  0.975  0.908  0.893
xgb   0.975  1.000  0.905  0.889
lr    0.908  0.905  1.000  0.984
svm   0.893  0.889  0.984  1.000


In [16]:
#stacking  - use lgbm classifier

#draw graph
#test on external datasets

#possible things to improve the model 
# compare model against validation score of each model-> drop weak models
#tune decision threshold








### Test on sarc_subreddit and sarc_temporal datasets



In [81]:
#Using equal weight voting on sarc_subreddit

# softvote_results, scores, weights = soft_vote(
#     val_sources={
#         'lgbm':       (subreddit_val_lgbm,   lgbm_model),
#         'xgboost':    (subreddit_val_lgbm,   xgb_model),
#         'lr':         (subreddit_val_logreg, lr_model),
#         'linear_svm': (subreddit_val_margin, svm_model),
#     },
#     test_sources={
#         'lgbm':       (subreddit_test_lgbm,   lgbm_model),
#         'xgboost':    (subreddit_test_lgbm,   xgb_model),   
#         'lr':         (subreddit_test_logreg, lr_model),
#         'linear_svm': (subreddit_test_margin, svm_model),
#     },
#     y_val=y_val,
#     id_path=test_logreg,
# )


print(len(pd.read_parquet(subreddit_val_lgbm)))
print(len(pd.read_parquet(subreddit_test_lgbm)))

135123
135534
